# LTX Video 2.3 22B – Modelle herunterladen

Dieses Notebook lädt alle benötigten Modell-Dateien von HuggingFace herunter.

**Gesamtgröße:** ~20–25 GB  
**Tipp:** Network Volume bei `/workspace/ComfyUI/models` einbinden – dann nur einmal herunterladen.

| Datei | Zielordner | Größe |
|---|---|---|
| `ltx-2.3-22b-distilled_transformer_only_fp8_input_scaled.safetensors` | `unet/` | ~10 GB |
| `gemma-3-12b-it-IQ4_XS.gguf` | `text_encoders/` | ~7 GB |
| `LTX-2.3-22b-distilled_embeddings_connectors.safetensors` | `text_encoders/` | ~1 GB |
| `LTX2.3-22b-distilled_audio_vae.safetensors` | `checkpoints/` | ~1 GB |
| `LTX-2.3-22b-distilled_video_vae.safetensors` | `vae/` | ~1 GB |
| `LTX-2.3-spatial-upscaler-x2-1.0.safetensors` | `latent_upscale_models/` | ~0.5 GB |

In [ ]:
import os
from pathlib import Path

# ── Konfiguration ──────────────────────────────────────────────────────────
MODEL_DIR = Path(os.environ.get('MODEL_DIR', '/workspace/ComfyUI/models'))
HF_BASE   = 'https://huggingface.co/Lightricks/LTX-Video/resolve/main'

# Optional: HuggingFace Token für private/gated Modelle
HF_TOKEN  = os.environ.get('HF_TOKEN', '')  # oder direkt eintragen

MODELS = [
    {
        'name':   'Diffusion Model (Transformer FP8)',
        'file':   'ltx-2.3-22b-distilled_transformer_only_fp8_input_scaled.safetensors',
        'folder': 'unet',
    },
    {
        'name':   'Text Encoder (Gemma 3 12B GGUF)',
        'file':   'gemma-3-12b-it-IQ4_XS.gguf',
        'folder': 'text_encoders',
    },
    {
        'name':   'Dual CLIP / Embeddings Connector',
        'file':   'LTX-2.3-22b-distilled_embeddings_connectors.safetensors',
        'folder': 'text_encoders',
    },
    {
        'name':   'Audio VAE',
        'file':   'LTX2.3-22b-distilled_audio_vae.safetensors',
        'folder': 'checkpoints',
    },
    {
        'name':   'Video VAE',
        'file':   'LTX-2.3-22b-distilled_video_vae.safetensors',
        'folder': 'vae',
    },
    {
        'name':   'Spatial Upscaler x2',
        'file':   'LTX-2.3-spatial-upscaler-x2-1.0.safetensors',
        'folder': 'latent_upscale_models',
    },
]

print(f'Modellverzeichnis: {MODEL_DIR}')
for m in MODELS:
    dest = MODEL_DIR / m['folder'] / m['file']
    status = '✅ vorhanden' if dest.exists() else '⬇️  fehlt'
    print(f'  {status}  {m["name"]}')

In [ ]:
import urllib.request
import time

def download_file(url: str, dest: Path, token: str = ''):
    """Lädt eine Datei herunter mit Fortschrittsanzeige."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    
    if dest.exists():
        print(f'  [skip] {dest.name} – bereits vorhanden ({dest.stat().st_size // 1024 // 1024} MB)')
        return
    
    headers = {}
    if token:
        headers['Authorization'] = f'Bearer {token}'
    
    req = urllib.request.Request(url, headers=headers)
    start = time.time()
    
    def progress(count, block_size, total_size):
        if total_size <= 0:
            return
        pct  = min(count * block_size / total_size * 100, 100)
        mb   = count * block_size / 1024 / 1024
        tot  = total_size / 1024 / 1024
        elapsed = time.time() - start
        speed = mb / elapsed if elapsed > 0 else 0
        print(f'\r  {pct:5.1f}%  {mb:.0f} / {tot:.0f} MB  ({speed:.1f} MB/s)  ', end='')
    
    print(f'  [download] {dest.name}')
    urllib.request.urlretrieve(url, dest, reporthook=progress)
    elapsed = time.time() - start
    print(f'\n  [ok] {dest.stat().st_size // 1024 // 1024} MB in {elapsed:.0f}s')


# ── Alle Modelle herunterladen ─────────────────────────────────────────────
for m in MODELS:
    url  = f"{HF_BASE}/{m['file']}"
    dest = MODEL_DIR / m['folder'] / m['file']
    print(f"\n{'─'*60}")
    print(f"  {m['name']}")
    download_file(url, dest, token=HF_TOKEN)

print(f"\n{'='*60}")
print('✅ Alle Downloads abgeschlossen!')

In [ ]:
# ── Dateien überprüfen ─────────────────────────────────────────────────────
print(f'Modellverzeichnis: {MODEL_DIR}\n')
total_mb = 0
for m in MODELS:
    dest = MODEL_DIR / m['folder'] / m['file']
    if dest.exists():
        mb = dest.stat().st_size / 1024 / 1024
        total_mb += mb
        print(f'  ✅  {m["name"]:45s}  {mb:7.0f} MB  →  {m["folder"]}/{dest.name}')
    else:
        print(f'  ❌  {m["name"]:45s}  FEHLT  →  {m["folder"]}/{dest.name}')

print(f'\nGesamt: {total_mb / 1024:.1f} GB')

In [ ]:
# ── ComfyUI neu starten damit neue Modelle erkannt werden ──────────────────
# (nur im Pod-Modus nötig, wenn ComfyUI bereits läuft)
import subprocess, requests

try:
    r = requests.get('http://127.0.0.1:8188/system_stats', timeout=3)
    if r.status_code == 200:
        # Modell-Cache neu einlesen ohne Neustart
        r2 = requests.post('http://127.0.0.1:8188/api/free', json={'unload_models': False, 'free_memory': False})
        print('✅ ComfyUI läuft – Modell-Liste wird neu geladen.')
        print('   Öffne ComfyUI und drücke F5 um die neuen Modelle zu sehen.')
except Exception:
    print('ℹ️  ComfyUI läuft nicht – Modelle werden beim nächsten Start erkannt.')